# INITIAL IMPORT

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
def parse_state_taxi(env, state):
    taxi_row, taxi_col, passenger_location, destination = env.unwrapped.decode(state)
    # print((taxi_row, taxi_col))
    return (taxi_row, taxi_col)

In [ ]:
import os
import numpy as np
import gymnasium as gym
from src.config import Configuration


CONFIG = Configuration(
    n_training_episodes = 150_000,
    learning_rate = 0.7,
    n_eval_episodes = 1000,
    # gym_id = "Taxi-v4",
    # gym_id = "MiniGrid-DoorKey-5x5-v0", 
    rm_file= "rm_taxi_2p.txt",
    # rm_file= "rm_taxi_v2.txt",
    max_steps = 1000,
    gamma = 0.99,
    max_epsilon = 1.0,
    min_epsilon = 0.05,
    decay_rate = 0.000001,
    use_rm = True, 
    use_crm = True, 
    parse_state = None, 
    # parse_state = parse_state_taxi,
    dynamic_qtable = True,
)

# LOAD ENV

In [ ]:
from src.envs import MultiTaxiEnv

env = MultiTaxiEnv(grid_size=5, num_passengers=2, render_mode='rgb_array')

state_space = env.observation_space.n
print("There are ", state_space, " possible states")
action_space = env.action_space.n
print("There are ", action_space, " possible actions")

state, _ = env.reset()
# Returns a tuple: (taxi_row, taxi_col, passenger_location, destination)
print(env.unwrapped.decode(state))

In [ ]:
import matplotlib.pyplot as plt
state, _ = env.reset()

env.render()

# 2. Get the single frame

img = env.render()

plt.figure(figsize=(5,5))
plt.imshow(img)
plt.axis('off')

T (Yellow): Empty Taxi

X (Green): Taxi carrying a passenger

P (Blue): Passenger waiting for pickup

D (Magenta): A passenger's destination location

# DEFINE Q-LEARNING

In [ ]:
from src.models import RewardMachine
from src.envs import get_propositions_multi_taxi

get_propositions_fn = get_propositions_multi_taxi

In [ ]:
from src.models import QTableRM

In [ ]:
from src.models import train_qtable_crm

In [ ]:
from src.models import evaluate_agent

# Generate and train

In [ ]:
rm = RewardMachine(CONFIG, CONFIG.rm_file)
rm.states

In [ ]:
rm.get_current_state()

In [ ]:
qt = QTableRM(CONFIG, env, rm_file=CONFIG.rm_file if CONFIG.use_rm else None, dynamic=CONFIG.dynamic_qtable)
qt = train_qtable_crm(
    CONFIG, 
    qt,  
    get_propositions_fn, 
    env,
)

In [ ]:
mean_reward, std_reward = evaluate_agent(CONFIG, qt, get_propositions_fn, env)
print(f"Mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")

In [ ]:
keys = sorted(qt._storage._table.keys())
print(f"Total Q-table entries: {len(keys)}")

In [ ]:
for key in keys:
    print(f"State: {key[0]}, RM: {key[1]}, Values: [{','.join(f'{i:.4f}' for i in qt._storage._table[key])}]")


# Record video

In [ ]:
from src.utils import record_video

max_videos = 5

for i in range(max_videos):
    record_video(CONFIG, qt, env, get_propositions_fn, video_name=f"{CONFIG.rm_file}-p-{i}.gif")


# Bench mark

In [ ]:

variants = [
    {
        "label": "No RM / No CRM", 
        "use_rm": False, 
        "use_crm": False, 
        'rm_file': None,
        'parse_state': None
    },
    {
        "label": "RM / No CRM", 
        "use_rm": True, 
        "use_crm": False, 
        'rm_file': 'rm_taxi_2p.txt',
        'parse_state': None
    },
    {
        "label": "RM / CRM", 
        "use_rm": True, 
        "use_crm": True, 
        'rm_file': 'rm_taxi_2p.txt',
        'parse_state': None
    },
    # {
    #     "label": "RM / No CRM NEW", 
    #     "use_rm": True, 
    #     "use_crm": False, 
    #     'rm_file': 'rm_taxi_v2.txt',
    #     # 'parse_state': None
    #     'parse_state': parse_state_taxi
    # },
    # {
    #     "label": "RM / CRM NEW", 
    #     "use_rm": True, 
    #     "use_crm": True, 
    #     'rm_file': 'rm_taxi_v2.txt',
    #     # 'parse_state': None
    #     'parse_state': parse_state_taxi
    # },
]

In [ ]:
import numpy as np
from typing import Callable
import matplotlib.pyplot as plt
from tqdm import tqdm

from src.config import Configuration
from src.models import QTableRM, evaluate_agent


def train_qtable_with_progress(config: Configuration, eval_every: int = 50):
    """Train and periodically evaluate to track learning evolution."""
    qtable = QTableRM(config, env, rm_file=config.rm_file if config.use_rm else None, dynamic=config.dynamic_qtable)

    iterations = []
    eval_means = []
    eval_stds = []

    for episode in tqdm(range(config.n_training_episodes), desc=f"RM={config.use_rm}, CRM={config.use_crm}"):
        epsilon = config.min_epsilon + (config.max_epsilon - config.min_epsilon) * np.exp(-config.decay_rate * episode)
        raw_state, _ = env.reset(seed=config.seed + episode)
        qtable.reset_rm()

        terminated, truncated = False, False
        state = config.parse_state(env, raw_state) if config.parse_state else raw_state
        
        for _ in range(config.max_steps):
            action = qtable.epsilon_greedy_policy(state, epsilon, env)
            new_state, env_reward, terminated, truncated, _ = env.step(action)

            new_state_parse = config.parse_state(env, new_state) if config.parse_state else new_state
            rm_done = qtable.update(
                state,
                action,
                env_reward,
                raw_state,
                new_state,
                new_state_parse,
                config.gamma,
                config.learning_rate,
                env,
                get_propositions=get_propositions_fn,
                terminated=terminated,
                use_crm=config.use_crm,
            )

            if terminated or truncated or rm_done:
                break

            raw_state = new_state
            state = new_state_parse

        if (episode + 1) % eval_every == 0 or episode == 0:
            mean_reward, std_reward = evaluate_agent(config, qtable, get_propositions_fn, env)
            iterations.append(episode + 1)
            eval_means.append(mean_reward)
            eval_stds.append(std_reward)

    return qtable, np.array(iterations), np.array(eval_means), np.array(eval_stds)


def convergence_iteration(iterations, values, smooth_window=3, plateau_ratio=0.95, patience=3):
    """Estimate convergence as first iteration that reaches and maintains near-plateau performance."""
    if len(values) == 0:
        return None

    if len(values) < smooth_window:
        smoothed = values
        smooth_iters = iterations
    else:
        kernel = np.ones(smooth_window) / smooth_window
        smoothed = np.convolve(values, kernel, mode="valid")
        smooth_iters = iterations[smooth_window - 1:]

    best = np.max(smoothed)
    threshold = best - (1.0 - plateau_ratio) * max(1.0, abs(best))

    if len(smoothed) < patience:
        return int(smooth_iters[np.argmax(smoothed)])

    for i in range(len(smoothed) - patience + 1):
        if np.all(smoothed[i : i + patience] >= threshold):
            return int(smooth_iters[i])

    return int(smooth_iters[np.argmax(smoothed)])


# Shared benchmark settings
base_kwargs = dict(
    n_training_episodes=150_000,
    learning_rate=0.7,
    n_eval_episodes=200,
    gym_id="Taxi-v4",
    max_steps=1000,
    gamma = 0.96,
    max_epsilon = 1.0,
    min_epsilon = 0.05,
    decay_rate = 0.0001,
)


results = {}

for variant in variants:
    cfg = Configuration(
        **base_kwargs,
        use_rm=variant["use_rm"],
        use_crm=variant["use_crm"],
        rm_file=variant["rm_file"],
        parse_state=variant['parse_state']
    )

    _, iters, means, stds = train_qtable_with_progress(cfg, eval_every=300)
    conv_it = convergence_iteration(iters, means, smooth_window=3, plateau_ratio=0.95, patience=3)

    results[variant["label"]] = {
        "iterations": iters,
        "mean_rewards": means,
        "std_rewards": stds,
        "convergence_iteration": conv_it,
    }


In [ ]:

# =============================================================
# =============================================================
import plotly.graph_objects as go

fig = go.Figure() 

for label, data in results.items():
    x = data["iterations"]
    y = data["mean_rewards"]
    s = data["std_rewards"]

    upper = y + s
    lower = y - s

    # Std shaded area
    fig.add_trace(
        go.Scatter(
            x=np.concatenate([x, x[::-1]]),
            y=np.concatenate([upper, lower[::-1]]),
            fill="toself",
            fillcolor="rgba(0,0,0,0.08)",
            line=dict(color="rgba(255,255,255,0)"),
            hoverinfo="skip",
            showlegend=False
        )
    )

    # Main line
    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            mode="lines+markers",
            name=label,
            line=dict(width=3),
            marker=dict(size=7),
            hovertemplate=(
                f"{label}<br>"
                "Episode: %{x}<br>"
                "Mean reward: %{y:.2f}<extra></extra>"
            )
        )
    )

fig.update_layout(
    title="Q-Learning Benchmark: Reward Evolution vs Training Episodes",
    xaxis_title="Training episodes",
    yaxis_title="Evaluation mean reward",
    template="plotly_white",
    width=1100,
    height=600,
    hovermode="x unified",
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=1.02
    )
)

fig.update_xaxes(showgrid=True, gridwidth=1)
fig.update_yaxes(showgrid=True, gridwidth=1)

fig.show()

# Human playing

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
from IPython.display import clear_output

# 1. Initialize with rgb_array for visual frames
if 'env' not in globals():
    env = gym.make(CONFIG.gym_id, render_mode="rgb_array")
    state, _ = env.reset()

# --- HUMAN PLAY ---
# 0=South, 1=North, 2=East, 3=West, 4=Pickup, 5=Dropoff
my_action = 5
# ------------------

# Step the simulation
state, reward, terminated, truncated, _ = env.step(my_action)

# 2. Get the single frame
img = env.render()

# 3. Visualize
clear_output(wait=True)
plt.figure(figsize=(5,5))
plt.imshow(img)
plt.axis('off')
plt.show()

# Data output
taxi_row, taxi_col, p_loc, dest = env.unwrapped.decode(state)
print(f"Action: {my_action} | Reward: {reward}")
print(f"Row: {taxi_row}, Col: {taxi_col}, P_Loc: {p_loc}, Dest: {dest}")

if terminated or truncated:
    print("Goal reached or Resetting...")
    state, _ = env.reset()